In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import ast
import re
from tqdm import tqdm
import os

# Load results
results = pd.read_csv("pgfgleam/all_results/local/full_results.csv")

# Parse vector columns
vector_cols = [
    'ICU_detection_times', 'ICU_local_cases_samples',
    'WW_008_10pct_detection_times', 'WW_008_10pct_local_cases_samples',
    'WW_008_25pct_detection_times', 'WW_008_25pct_local_cases_samples',
    'WW_008_50pct_detection_times', 'WW_008_50pct_local_cases_samples',
    'WW_008_100pct_detection_times', 'WW_008_100pct_local_cases_samples',
    'WW_016_10pct_detection_times', 'WW_016_10pct_local_cases_samples',
    'WW_016_25pct_detection_times', 'WW_016_25pct_local_cases_samples',
    'WW_016_50pct_detection_times', 'WW_016_50pct_local_cases_samples',
    'WW_016_100pct_detection_times', 'WW_016_100pct_local_cases_samples'
]

def parse_julia_vector(s):
    if pd.isna(s): return []
    s = str(s).strip()
    if re.match(r'^[A-Za-z0-9]+\[\]$', s): return []
    try:
        return ast.literal_eval(s)
    except:
        try:
            numbers = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)
            return [float(n) for n in numbers]
        except:
            return []

for col in vector_cols:
    results[col] = results[col].apply(parse_julia_vector)

def hierarchical_bootstrap_ci(subset, sample_col, n_bootstrap=100000):
    bootstrap_samples = []
    valid_countries = []
    country_samples = []
    for idx, row in subset.iterrows():
        samples = row[sample_col]
        valid_samples = [s for s in samples if pd.notna(s) and np.isfinite(s)]
        if len(valid_samples) > 0:
            valid_countries.append(idx)
            country_samples.append(valid_samples)
    if len(valid_countries) == 0:
        return {'mean': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan}
    for _ in range(n_bootstrap):
        country_idx = np.random.choice(len(valid_countries))
        value = np.random.choice(country_samples[country_idx])
        bootstrap_samples.append(value)
    return {
        'mean': np.mean(bootstrap_samples),
        'ci_lower': np.percentile(bootstrap_samples, 2.5),
        'ci_upper': np.percentile(bootstrap_samples, 97.5)
    }

def simple_mean_ci(subset, mean_col):
    valid_values = [v for v in subset[mean_col] if pd.notna(v) and np.isfinite(v)]
    if not valid_values:
        return {'mean': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan}
    mean_val = np.mean(valid_values)
    se = stats.sem(valid_values)
    return {
        'mean': mean_val,
        'ci_lower': mean_val - 1.96 * se,
        'ci_upper': mean_val + 1.96 * se
    }

combinations = results[['R0', 'gen_time']].drop_duplicates().sort_values(['R0', 'gen_time'])
summary_data = []

for _, combo in tqdm(list(combinations.iterrows()), desc="Processing combinations"):
    R0_v, GT_v = combo['R0'], combo['gen_time']
    sub = results[(results['R0'] == R0_v) & (results['gen_time'] == GT_v)]
    res = {'R0': R0_v, 'gen_time': GT_v}
    
    res['ICU_t_s'] = simple_mean_ci(sub, 'ICU_mean_detection_time')
    res['ICU_t_h'] = hierarchical_bootstrap_ci(sub, 'ICU_detection_times')
    res['ICU_c_s'] = simple_mean_ci(sub, 'ICU_mean_local_cases')
    res['ICU_c_h'] = hierarchical_bootstrap_ci(sub, 'ICU_local_cases_samples')
    
    ww_ids = ['008_10pct', '008_25pct', '008_50pct', '008_100pct', 
              '016_10pct', '016_25pct', '016_50pct', '016_100pct']
    for wid in ww_ids:
        res[f'WW_{wid}_t_s'] = simple_mean_ci(sub, f'WW_{wid}_mean_detection_time')
        res[f'WW_{wid}_t_h'] = hierarchical_bootstrap_ci(sub, f'WW_{wid}_detection_times')
        res[f'WW_{wid}_c_s'] = simple_mean_ci(sub, f'WW_{wid}_mean_local_cases')
        res[f'WW_{wid}_c_h'] = hierarchical_bootstrap_ci(sub, f'WW_{wid}_local_cases_samples')
    summary_data.append(res)

# ========================================================================
# EXPORT TO INDIVIDUAL CSV FILES FOR PERFECT GOOGLE SHEETS IMPORT
# ========================================================================
def fmt_cell(data):
    if np.isnan(data['mean']): return "N/A"
    return f"{data['mean']:.1f}: ({data['ci_lower']:.1f}, {data['ci_upper']:.1f})"

output_dir = "formatted_tables"
os.makedirs(output_dir, exist_ok=True)

for table in summary_data:
    filename = f"{output_dir}/Table_R0_{table['R0']}_GT_{table['gen_time']}.csv"
    
    rows = []
    rows.append(["Method", "p_det", "Mean Detection Time (95% CI)", "Bootstrap (95% PI)", "EW Local Cases (95% CI)", "Bootstrap (95% PI)"])
    rows.append(["ICU", " ", fmt_cell(table['ICU_t_s']), fmt_cell(table['ICU_t_h']), fmt_cell(table['ICU_c_s']), fmt_cell(table['ICU_c_h'])])
    
    configs = [
        ('008_10pct', 'AWW (10%)', '8%'), ('008_25pct', 'AWW (25%)', '8%'),
        ('008_50pct', 'AWW (50%)', '8%'), ('008_100pct', 'AWW (100%)', '8%'),
        ('016_10pct', 'AWW (10%)', '16%'), ('016_25pct', 'AWW (25%)', '16%'),
        ('016_50pct', 'AWW (50%)', '16%'), ('016_100pct', 'AWW (100%)', '16%')
    ]
    
    for wid, label, p_det in configs:
        rows.append([
            label, p_det, 
            fmt_cell(table[f'WW_{wid}_t_s']), fmt_cell(table[f'WW_{wid}_t_h']),
            fmt_cell(table[f'WW_{wid}_c_s']), fmt_cell(table[f'WW_{wid}_c_h'])
        ])
    
    # Save as actual CSV
    df_table = pd.DataFrame(rows[1:], columns=rows[0])
    df_table.to_csv(filename, index=False)
    print(f"Saved: {filename}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the comparison results
comparison = pd.read_csv("pgfgleam/all_results/global/comparison_simple_vs_hierarchical_CIs.csv")

# Define only the 16% base configurations with 25% and 50% sampling
ww_configs = [
    ('016_25pct', '16% base, 25% sampling', 0.040),
    ('016_50pct', '16% base, 50% sampling', 0.080),
]

def ci_to_se(ci_lower, ci_upper):
    """Convert 95% CI bounds to standard error."""
    return (ci_upper - ci_lower) / 3.92

def se_to_ci(mean, se):
    """Convert mean and SE to 95% CI bounds."""
    return mean - 1.96 * se, mean + 1.96 * se

# Compute differences and ratios with propagated CIs
for config_name, config_label, eff_pdet in ww_configs:
    ww_prefix = f'WW_{config_name}'

    # --- Means ---
    comparison[f'ICU_minus_{ww_prefix}_time_simple'] = (
        comparison['ICU_time_simple_mean'] - comparison[f'{ww_prefix}_time_simple_mean']
    )
    comparison[f'ICU_minus_{ww_prefix}_cases_simple'] = (
        comparison['ICU_cases_simple_mean'] - comparison[f'{ww_prefix}_cases_simple_mean']
    )
    comparison[f'ICU_div_{ww_prefix}_cases_simple'] = (
        comparison['ICU_cases_simple_mean'] / comparison[f'{ww_prefix}_cases_simple_mean']
    )

    # --- Standard errors from stored CIs ---
    se_icu_time   = ci_to_se(comparison['ICU_time_simple_ci_lower'],   comparison['ICU_time_simple_ci_upper'])
    se_ww_time    = ci_to_se(comparison[f'{ww_prefix}_time_simple_ci_lower'],  comparison[f'{ww_prefix}_time_simple_ci_upper'])
    se_icu_cases  = ci_to_se(comparison['ICU_cases_simple_ci_lower'],  comparison['ICU_cases_simple_ci_upper'])
    se_ww_cases   = ci_to_se(comparison[f'{ww_prefix}_cases_simple_ci_lower'], comparison[f'{ww_prefix}_cases_simple_ci_upper'])

    # --- Propagated CIs for differences ---
    se_time_diff  = np.sqrt(se_icu_time**2  + se_ww_time**2)
    se_cases_diff = np.sqrt(se_icu_cases**2 + se_ww_cases**2)

    comparison[f'ICU_minus_{ww_prefix}_time_ci_lower'],  comparison[f'ICU_minus_{ww_prefix}_time_ci_upper']  = \
        se_to_ci(comparison[f'ICU_minus_{ww_prefix}_time_simple'],  se_time_diff)
    comparison[f'ICU_minus_{ww_prefix}_cases_ci_lower'], comparison[f'ICU_minus_{ww_prefix}_cases_ci_upper'] = \
        se_to_ci(comparison[f'ICU_minus_{ww_prefix}_cases_simple'], se_cases_diff)

    # --- Propagated CIs for ratio via delta method: SE(A/B) ≈ (A/B)*sqrt((SE_A/A)^2 + (SE_B/B)^2) ---
    mu_icu   = comparison['ICU_cases_simple_mean']
    mu_ww    = comparison[f'{ww_prefix}_cases_simple_mean']
    ratio    = comparison[f'ICU_div_{ww_prefix}_cases_simple']
    se_ratio = ratio * np.sqrt((se_icu_cases / mu_icu)**2 + (se_ww_cases / mu_ww)**2)

    comparison[f'ICU_div_{ww_prefix}_cases_ci_lower'], comparison[f'ICU_div_{ww_prefix}_cases_ci_upper'] = \
        se_to_ci(ratio, se_ratio)


# ── Detailed results ──────────────────────────────────────────────────────────
print("\n" + "="*110)
print("DETAILED RESULTS BY PARAMETER COMBINATION")
print("="*110)

for _, row in comparison.iterrows():
    print(f"\nR0 = {row['R0']}, Generation Time = {row['gen_time']} days (n={row['n_countries']} countries)")
    print("-" * 110)

    for config_name, config_label, eff_pdet in ww_configs:
        ww_prefix = f'WW_{config_name}'

        time_diff  = row[f'ICU_minus_{ww_prefix}_time_simple']
        time_lo    = row[f'ICU_minus_{ww_prefix}_time_ci_lower']
        time_hi    = row[f'ICU_minus_{ww_prefix}_time_ci_upper']

        cases_diff = row[f'ICU_minus_{ww_prefix}_cases_simple']
        cases_lo   = row[f'ICU_minus_{ww_prefix}_cases_ci_lower']
        cases_hi   = row[f'ICU_minus_{ww_prefix}_cases_ci_upper']

        ratio      = row[f'ICU_div_{ww_prefix}_cases_simple']
        ratio_lo   = row[f'ICU_div_{ww_prefix}_cases_ci_lower']
        ratio_hi   = row[f'ICU_div_{ww_prefix}_cases_ci_upper']

        print(f"  {config_label} (effective p_det={eff_pdet}):")
        print(f"    Detection Time Difference  (ICU - WW): {time_diff:+.2f} days  "
              f"[95% CI: {time_lo:+.2f}, {time_hi:+.2f}]")
        print(f"    Local Cases Difference     (ICU - WW): {cases_diff:+.1f} cases "
              f"[95% CI: {cases_lo:+.1f}, {cases_hi:+.1f}]")
        print(f"    Local Cases Ratio          (ICU / WW): {ratio:.2f}x              "
              f"[95% CI: {ratio_lo:.2f}x, {ratio_hi:.2f}x]")


# ── Summary statistics ────────────────────────────────────────────────────────
print("\n" + "="*110)
print("SUMMARY STATISTICS ACROSS ALL R0/GENERATION TIME COMBINATIONS")
print("="*110)

for config_name, config_label, eff_pdet in ww_configs:
    ww_prefix = f'WW_{config_name}'

    time_diff_col  = f'ICU_minus_{ww_prefix}_time_simple'
    cases_diff_col = f'ICU_minus_{ww_prefix}_cases_simple'
    cases_ratio_col = f'ICU_div_{ww_prefix}_cases_simple'

    print(f"\n{config_label} (effective p_det={eff_pdet}):")

    print(f"  Detection Time Difference (ICU - WW):")
    print(f"    Range:   [{comparison[time_diff_col].min():+.2f}, {comparison[time_diff_col].max():+.2f}] days")
    print(f"    Mean ± SD: {comparison[time_diff_col].mean():+.2f} ± {comparison[time_diff_col].std():.2f} days")

    print(f"  Local Cases Difference (ICU - WW):")
    print(f"    Range:   [{comparison[cases_diff_col].min():+.1f}, {comparison[cases_diff_col].max():+.1f}] cases")
    print(f"    Mean ± SD: {comparison[cases_diff_col].mean():+.1f} ± {comparison[cases_diff_col].std():.1f} cases")

    print(f"  Local Cases Ratio (ICU / WW):")
    print(f"    Range:   [{comparison[cases_ratio_col].min():.2f}x, {comparison[cases_ratio_col].max():.2f}x]")
    print(f"    Mean ± SD: {comparison[cases_ratio_col].mean():.2f}x ± {comparison[cases_ratio_col].std():.2f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the comparison results
comparison = pd.read_csv("pgfgleam/all_results/global/comparison_simple_vs_hierarchical_CIs.csv")

# Define only the 16% base configurations with 25% and 50% sampling
ww_configs = [
    ('016_25pct', '16% base, 25% sampling', 0.040),
    ('016_50pct', '16% base, 50% sampling', 0.080),
]

# Compute differences and ratios for selected configurations
for config_name, config_label, eff_pdet in ww_configs:
    time_diff_col = f'ICU_minus_WW{config_name}_time_simple'
    cases_diff_col = f'ICU_minus_WW{config_name}_cases_simple'
    cases_ratio_col = f'ICU_div_WW{config_name}_cases_simple'
    
    comparison[time_diff_col] = comparison['ICU_time_simple_mean'] - comparison[f'WW_{config_name}_time_simple_mean']
    comparison[cases_diff_col] = comparison['ICU_cases_simple_mean'] - comparison[f'WW_{config_name}_cases_simple_mean']
    comparison[cases_ratio_col] = comparison['ICU_cases_simple_mean'] / comparison[f'WW_{config_name}_cases_simple_mean']

# Print detailed results for each R0/gen_time combination
print("\n" + "="*100)
print("DETAILED RESULTS BY PARAMETER COMBINATION")
print("="*100)

for _, row in comparison.iterrows():
    print(f"\nR0 = {row['R0']}, Generation Time = {row['gen_time']} days (n={row['n_countries']} countries)")
    print("-" * 100)
    
    for config_name, config_label, eff_pdet in ww_configs:
        time_diff = row[f'ICU_minus_WW{config_name}_time_simple']
        cases_diff = row[f'ICU_minus_WW{config_name}_cases_simple']
        cases_ratio = row[f'ICU_div_WW{config_name}_cases_simple']
        
        print(f"  {config_label} (effective p_det={eff_pdet}):")
        print(f"    Detection Time Difference (ICU - WW): {time_diff:+.2f} days")
        print(f"    Local Cases Difference (ICU - WW): {cases_diff:+.1f} cases")
        print(f"    Local Cases Ratio (ICU / WW): {cases_ratio:.2f}x")

# Print summary statistics
print("\n" + "="*100)
print("SUMMARY STATISTICS ACROSS ALL R0/GENERATION TIME COMBINATIONS")
print("="*100)

for config_name, config_label, eff_pdet in ww_configs:
    time_diff_col = f'ICU_minus_WW{config_name}_time_simple'
    cases_diff_col = f'ICU_minus_WW{config_name}_cases_simple'
    cases_ratio_col = f'ICU_div_WW{config_name}_cases_simple'
    
    print(f"\n{config_label} (effective p_det={eff_pdet}):")
    print(f"  Detection Time Difference (ICU - WW):")
    print(f"    Range: [{comparison[time_diff_col].min():+.2f}, {comparison[time_diff_col].max():+.2f}] days")
    print(f"    Mean ± SD: {comparison[time_diff_col].mean():+.2f} ± {comparison[time_diff_col].std():.2f} days")
    
    print(f"  Local Cases Difference (ICU - WW):")
    print(f"    Range: [{comparison[cases_diff_col].min():+.1f}, {comparison[cases_diff_col].max():+.1f}] cases")
    print(f"    Mean ± SD: {comparison[cases_diff_col].mean():+.1f} ± {comparison[cases_diff_col].std():.1f} cases")
    
    print(f"  Local Cases Ratio (ICU / WW):")
    print(f"    Range: [{comparison[cases_ratio_col].min():.2f}x, {comparison[cases_ratio_col].max():.2f}x]")
    print(f"    Mean ± SD: {comparison[cases_ratio_col].mean():.2f}x ± {comparison[cases_ratio_col].std():.2f}")
